# Quantum SAM continual semantic segmentation

This notebook performs the full Colab setup: clone, dependencies, Kaggle download, layout preparation, smoke test, and sequential OpenEarthMap → LoveDA training. Enable a GPU in **Runtime → Change runtime type** first.

In [ ]:
!git clone https://github.com/KAVINRAJ06/Quantum_CL.git
%cd Quantum_CL
!python -m pip install -q -r requirements-colab.txt
!python -c "import torch, pennylane, transformers; print('torch=', torch.__version__, 'cuda=', torch.cuda.is_available(), 'pennylane=', pennylane.__version__, 'transformers=', transformers.__version__)


## Kaggle credentials
Upload the `kaggle.json` API token downloaded from Kaggle Settings. It is stored only in this Colab runtime.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'kaggle.json' in uploaded, 'Upload kaggle.json from Kaggle Settings > API'
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!mkdir -p raw/openearthmap raw/loveda
!kaggle datasets download -d aletbm/global-land-cover-mapping-openearthmap -p raw/openearthmap --unzip
!kaggle datasets download -d mohammedjaveed/loveda-dataset -p raw/loveda --unzip
!python scripts/prepare_remote_sensing_data.py --source raw/openearthmap --destination data/openearthmap
!python scripts/prepare_remote_sensing_data.py --source raw/loveda --destination data/loveda
!find data -type f | head -20


## Confirm masks
The training loader requires indexed masks. If the previous step reports RGB masks, use the verified palette for that Kaggle mirror with `scripts/convert_color_masks.py` before continuing. Do not guess palette colours.

In [ ]:
!python smoke_test.py
!python train.py --data-root data --tasks openearthmap loveda --num-classes 8 --sam-model facebook/sam-vit-base --image-size 512 --batch-size 2 --epochs-per-task 15 --qubits 8 --freeze-sam
